## **NOTEBOOK 01: DATA PREPROCESSING & CLEANING - FIXED VERSION**

**Air Quality Monitoring - European Cities**

# CELL 1: IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import warnings
import os
import glob
warnings.filterwarnings('ignore')

# FIX: Agar chart tampil di output Google Colab
pio.renderers.default = 'colab'

print("✅ Libraries loaded")

✅ Libraries loaded


# CELL 2: MOUNT DRIVE & SETUP PATHS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_PATH            = '/content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities'
RAW_DATA_PATH        = f'{BASE_PATH}/data/raw'
PROCESSED_DATA_PATH  = f'{BASE_PATH}/data/processed'
CLEANED_DATA_PATH    = f'{BASE_PATH}/data/cleaned'
RESULTS_PATH         = f'{BASE_PATH}/results'
RESULTS_PREPROCESSING = f'{RESULTS_PATH}/preprocessing'

os.makedirs(PROCESSED_DATA_PATH,   exist_ok=True)
os.makedirs(CLEANED_DATA_PATH,     exist_ok=True)
os.makedirs(RESULTS_PREPROCESSING, exist_ok=True)

print(f"✅ Google Drive connected")
print(f"✅ Base path: {BASE_PATH}")

✅ Google Drive connected
✅ Base path: /content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities


# CELL 3: LOAD DATASETS

In [ ]:
csv_files = glob.glob(f'{RAW_DATA_PATH}/**/*.csv', recursive=True)
print(f"\n📂 Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"   • {f.split('/')[-1]}")

data_ancona   = pd.read_csv(f'{RAW_DATA_PATH}/ancona_data.csv')
data_athens   = pd.read_csv(f'{RAW_DATA_PATH}/athens_data.csv')
data_zaragoza = pd.read_csv(f'{RAW_DATA_PATH}/zaragoza_data.csv')

print(f"\n✅ Datasets loaded")
print(f"   Ancona   shape: {data_ancona.shape}")
print(f"   Athens   shape: {data_athens.shape}")
print(f"   Zaragoza shape: {data_zaragoza.shape}")


📂 Found 3 CSV files:
   • ancona_data.csv
   • athens_data.csv
   • zaragoza_data.csv

✅ Datasets loaded
   Ancona   shape: (417626, 19)
   Athens   shape: (1726464, 19)
   Zaragoza shape: (238336, 16)


# CELL 4: DATA EXPLORATION — LIHAT KOLOM ASLI

In [ ]:
print("\n📋 ANCONA - Column Names:")
print(data_ancona.columns.tolist())
print("\n📋 ATHENS - Column Names:")
print(data_athens.columns.tolist())
print("\n📋 ZARAGOZA - Column Names:")
print(data_zaragoza.columns.tolist())


📋 ANCONA - Column Names:
['Date', 'NO2', 'O3', 'PM10', 'PM2.5', 'Latitude', 'Longitude', 'station_name', 'Wind-Speed (U)', 'Wind-Speed (V)', 'Dewpoint Temp', 'Soil Temp', 'Total Percipitation', 'Vegitation (High)', 'Vegitation (Low)', 'Temp', 'Relative Humidity', 'code', 'id']

📋 ATHENS - Column Names:
['Date', 'Latitude', 'Longitude', 'station_name', 'Wind-Speed (U)', 'Wind-Speed (V)', 'Dewpoint Temp', 'Soil Temp', 'Total Percipitation', 'Vegitation (High)', 'Vegitation (Low)', 'Temp', 'Relative Humidity', 'PM10', 'PM2.5', 'NO2', 'O3', 'code', 'id']

📋 ZARAGOZA - Column Names:
['Date', 'NO2', 'O3', 'PM10', 'Latitude', 'Longitude', 'station_name', 'Wind-Speed (U)', 'Wind-Speed (V)', 'Dewpoint Temp', 'Temp', 'Vegitation (High)', 'Vegitation (Low)', 'Soil Temp', 'Total Percipitation', 'Relative Humidity']


In [ ]:
# Identifikasi perbedaan kolom antar kota
all_cols = set(data_ancona.columns) | set(data_athens.columns) | set(data_zaragoza.columns)
print("\n📊 PERBEDAAN KOLOM ANTAR KOTA:")
for col in sorted(all_cols):
    in_ancona   = '✅' if col in data_ancona.columns   else '❌'
    in_athens   = '✅' if col in data_athens.columns   else '❌'
    in_zaragoza = '✅' if col in data_zaragoza.columns else '❌'
    if not (col in data_ancona.columns and
            col in data_athens.columns and
            col in data_zaragoza.columns):
        print(f"   {col:<30} Ancona:{in_ancona}  Athens:{in_athens}  Zaragoza:{in_zaragoza}")


📊 PERBEDAAN KOLOM ANTAR KOTA:
   PM2.5                          Ancona:✅  Athens:✅  Zaragoza:❌
   code                           Ancona:✅  Athens:✅  Zaragoza:❌
   id                             Ancona:✅  Athens:✅  Zaragoza:❌


# CELL 5: STANDARDIZE COLUMN NAMES

In [ ]:
column_mapping = {
    # Date
    'Date'                : 'date',
    'date'                : 'date',
    # Pollutants
    'NO2'                 : 'no2',
    'O3'                  : 'o3',
    'PM10'                : 'pm10',
    'PM2.5'               : 'pm25',       # Zaragoza tidak punya ini — tidak masalah
    # Location
    'Latitude'            : 'latitude',
    'latitude'            : 'latitude',
    'Longitude'           : 'longitude',
    'longitude'           : 'longitude',
    'station_name'        : 'station_name',
    'Station name'        : 'station_name',
    # Meteorologi
    'Wind-Speed (U)'      : 'wind_u',
    'Wind-Speed (V)'      : 'wind_v',
    'Dewpoint Temp'       : 'dewpoint_temp',
    'Temp'                : 'temperature',
    'temperature'         : 'temperature',
    'Relative Humidity'   : 'relative_humidity',
    'Soil Temp'           : 'soil_temp',
    'Total Percipitation' : 'precipitation',
    'Vegitation (High)'   : 'vegetation_high',
    'Vegitation (Low)'    : 'vegetation_low',
    # Identifiers
    'code'                : 'code',
    'id'                  : 'id',
}

In [ ]:
data_ancona.rename(columns=column_mapping,   inplace=True)
data_athens.rename(columns=column_mapping,   inplace=True)
data_zaragoza.rename(columns=column_mapping, inplace=True)

In [ ]:
# Tambah city column
data_ancona['city']   = 'Ancona'
data_athens['city']   = 'Athens'
data_zaragoza['city'] = 'Zaragoza'

print("✅ Column names standardized")

✅ Column names standardized


# CELL 6: HANDLE KOLOM YANG TIDAK ADA DI ZARAGOZA

FIX: PM2.5, code, id tidak ada di Zaragoza

In [ ]:
# PM2.5: Zaragoza memang tidak punya data ini dari sumber
# → Jangan diisi/diinterpolasi! Biarkan NaN agar konteks analisis terjaga
if 'pm25' not in data_zaragoza.columns:
    print("⚠️  Zaragoza: kolom PM2.5 tidak tersedia di dataset asli")
    print("   → Kolom pm25 akan bernilai NaN setelah merge")
    print("   → TIDAK akan diinterpolasi (data memang tidak tersedia)")

# code: isi placeholder string agar struktur konsisten
if 'code' not in data_zaragoza.columns:
    data_zaragoza['code'] = 'ZGZ_UNKNOWN'
    print("⚠️  Zaragoza: kolom 'code' tidak ada → diisi 'ZGZ_UNKNOWN'")

# id: isi placeholder integer
if 'id' not in data_zaragoza.columns:
    data_zaragoza['id'] = -1
    print("⚠️  Zaragoza: kolom 'id' tidak ada → diisi -1")

# FIX: Tambah flag pm25_available — penting untuk semua notebook berikutnya
# Notebook 02-09 akan pakai flag ini untuk skip/include Zaragoza di analisis PM2.5
data_ancona['pm25_available']   = True
data_athens['pm25_available']   = True
data_zaragoza['pm25_available'] = False

print("\n✅ pm25_available flag ditambahkan:")
print("   Ancona   → pm25_available = True")
print("   Athens   → pm25_available = True")
print("   Zaragoza → pm25_available = False")

⚠️  Zaragoza: kolom PM2.5 tidak tersedia di dataset asli
   → Kolom pm25 akan bernilai NaN setelah merge
   → TIDAK akan diinterpolasi (data memang tidak tersedia)
⚠️  Zaragoza: kolom 'code' tidak ada → diisi 'ZGZ_UNKNOWN'
⚠️  Zaragoza: kolom 'id' tidak ada → diisi -1

✅ pm25_available flag ditambahkan:
   Ancona   → pm25_available = True
   Athens   → pm25_available = True
   Zaragoza → pm25_available = False


# CELL 7: PARSE DATETIME & EXTRACT TEMPORAL FEATURES

In [ ]:
for df in [data_ancona, data_athens, data_zaragoza]:
    df['date']      = pd.to_datetime(df['date'], errors='coerce')
    df['year']      = df['date'].dt.year
    df['month']     = df['date'].dt.month
    df['day']       = df['date'].dt.day
    df['hour']      = df['date'].dt.hour
    df['dayofweek'] = df['date'].dt.dayofweek
    df['quarter']   = df['date'].dt.quarter

print("✅ Datetime parsed & temporal features extracted")
print(f"   Ancona   date range: {data_ancona['date'].min()} → {data_ancona['date'].max()}")
print(f"   Athens   date range: {data_athens['date'].min()} → {data_athens['date'].max()}")
print(f"   Zaragoza date range: {data_zaragoza['date'].min()} → {data_zaragoza['date'].max()}")

✅ Datetime parsed & temporal features extracted
   Ancona   date range: 2021-09-01 01:00:00 → 2023-10-31 23:00:00
   Athens   date range: 2020-05-01 00:00:00 → 2023-05-29 23:00:00
   Zaragoza date range: 2020-08-01 01:00:00 → 2023-05-29 23:00:00


# CELL 8: MISSING VALUES ANALYSIS (SEBELUM HANDLING)

In [ ]:
def analyze_missing_values(df, city_name):
    missing_count = df.isnull().sum()
    missing_pct   = (df.isnull().sum() / len(df)) * 100
    missing_df = pd.DataFrame({
        'Column'            : missing_count.index,
        'Missing_Count'     : missing_count.values,
        'Missing_Percentage': missing_pct.values
    }).sort_values('Missing_Percentage', ascending=False)
    return missing_df[missing_df['Missing_Count'] > 0]

missing_ancona   = analyze_missing_values(data_ancona,   'Ancona')
missing_athens   = analyze_missing_values(data_athens,   'Athens')
missing_zaragoza = analyze_missing_values(data_zaragoza, 'Zaragoza')

print("📊 MISSING VALUES - ANCONA:")
print(missing_ancona.to_string())
print("\n📊 MISSING VALUES - ATHENS:")
print(missing_athens.to_string())
print("\n📊 MISSING VALUES - ZARAGOZA:")
print(missing_zaragoza.to_string())

📊 MISSING VALUES - ANCONA:
  Column  Missing_Count  Missing_Percentage
4   pm25          45118           10.803446
3   pm10          27859            6.670801
2     o3          24803            5.939046
1    no2           7778            1.862432

📊 MISSING VALUES - ATHENS:
               Column  Missing_Count  Missing_Percentage
16                 o3         148064            8.576142
15                no2         143293            8.299797
17               code          26976            1.562500
9     vegetation_high           8573            0.496564
8       precipitation           8573            0.496564
10     vegetation_low           8573            0.496564
7           soil_temp           8573            0.496564
12  relative_humidity           8573            0.496564
6       dewpoint_temp           8573            0.496564
5              wind_v           8573            0.496564
4              wind_u           8573            0.496564
11        temperature           7565     

# CELL 9: VISUALISASI MISSING VALUES (TAMPIL DI COLAB + SIMPAN KE DRIVE)

In [ ]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Ancona', 'Athens', 'Zaragoza'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}]]
)

for idx, (df, city) in enumerate(
    [(data_ancona, 'Ancona'), (data_athens, 'Athens'), (data_zaragoza, 'Zaragoza')], 1
):
    missing = (df.isnull().sum() / len(df)) * 100
    missing = missing[missing > 0].sort_values(ascending=False)

    if len(missing) == 0:
        continue

    colors = ['#FF6B6B' if x > 30 else '#FFA07A' if x > 10 else '#FFD700'
              for x in missing.values]

    fig.add_trace(
        go.Bar(
            x=missing.values,
            y=missing.index,
            orientation='h',
            marker=dict(color=colors),
            name=city,
            showlegend=False,
            text=[f'{v:.1f}%' for v in missing.values],
            textposition='outside',
            hovertemplate='<b>%{y}</b><br>Missing: %{x:.1f}%<extra></extra>'
        ),
        row=1, col=idx
    )
    fig.update_xaxes(title_text='Missing %', row=1, col=idx)

fig.update_layout(
    title_text='Missing Values Analysis per City (Before Handling)',
    title_font_size=20,
    title_font_color='#1f77b4',
    height=500,
    showlegend=False,
    hovermode='closest',
    plot_bgcolor='rgba(240,240,240,0.5)',
    paper_bgcolor='white',
    margin=dict(l=150, r=80, t=80, b=60)
)

fig.write_html(f'{RESULTS_PREPROCESSING}/01_missing_values_analysis.html')
print("✅ Saved: 01_missing_values_analysis.html")
fig.show()  # Tampil di Colab output

✅ Saved: 01_missing_values_analysis.html


# CELL 10: HANDLE MISSING VALUES

FIX: deprecated ffill/bfill + skip PM2.5 Zaragoza

In [ ]:
def handle_missing_values(df, city_name):
    print(f"\n🔧 Processing {city_name}...")
    df_filled = df.copy()

    # --- Meteorologi: ffill max 24h + linear interpolation ---
    meteo_cols = ['temperature', 'relative_humidity', 'dewpoint_temp', 'soil_temp']
    for col in meteo_cols:
        if col not in df_filled.columns:
            continue
        before = df_filled[col].isnull().sum()
        # FIX: ffill() bukan fillna(method='ffill') — sudah deprecated
        df_filled[col] = df_filled[col].ffill(limit=24)
        df_filled[col] = df_filled[col].interpolate(method='linear', limit_direction='both')
        after = df_filled[col].isnull().sum()
        print(f"   ✓ {col}: {before} NaN → {after} NaN (ffill + interpolate)")

    # --- Wind: ffill + interpolation ---
    for col in ['wind_u', 'wind_v']:
        if col not in df_filled.columns:
            continue
        before = df_filled[col].isnull().sum()
        df_filled[col] = df_filled[col].ffill(limit=24)
        df_filled[col] = df_filled[col].interpolate(method='linear', limit_direction='both')
        after = df_filled[col].isnull().sum()
        print(f"   ✓ {col}: {before} NaN → {after} NaN (ffill + interpolate)")

    # --- Polutan: linear interpolation ---
    # FIX: Skip PM2.5 Zaragoza — kalau semua NaN berarti data memang tidak tersedia
    pollutant_cols = ['pm25', 'pm10', 'no2', 'o3']
    for col in pollutant_cols:
        if col not in df_filled.columns:
            continue

        before = df_filled[col].isnull().sum()

        # Skip jika seluruh kolom memang NaN (contoh: PM2.5 Zaragoza)
        if df_filled[col].isnull().all():
            print(f"   ⚠ {col}: semua NaN (data tidak tersedia di {city_name}) → SKIP")
            continue

        df_filled[col] = df_filled[col].interpolate(method='linear', limit_direction='both')
        after = df_filled[col].isnull().sum()
        print(f"   ✓ {col}: {before} NaN → {after} NaN (linear interpolate)")

    # --- Precipitation: fill 0 ---
    if 'precipitation' in df_filled.columns:
        before = df_filled['precipitation'].isnull().sum()
        df_filled['precipitation'] = df_filled['precipitation'].fillna(0)
        after = df_filled['precipitation'].isnull().sum()
        print(f"   ✓ precipitation: {before} NaN → {after} NaN (fill with 0)")

    # --- Vegetation: fill median jika <50% missing ---
    for col in ['vegetation_high', 'vegetation_low']:
        if col not in df_filled.columns:
            continue
        missing_pct = (df_filled[col].isnull().sum() / len(df_filled)) * 100
        if missing_pct < 50:
            median_val = df_filled[col].median()
            before = df_filled[col].isnull().sum()
            df_filled[col] = df_filled[col].fillna(median_val)
            after = df_filled[col].isnull().sum()
            print(
                f"   ✓ {col}: {before} NaN → {after} NaN "
                f"(filled with median = {median_val:.4f})"
            )
        else:
            print(f"   ⚠ {col}: {missing_pct:.1f}% missing → terlalu banyak, skip")

    # --- Identifier Columns ---
    # code
    if 'code' in df_filled.columns:
        before = df_filled['code'].isnull().sum()
        df_filled['code'] = df_filled['code'].fillna(
            f"{city_name.upper()}_UNKNOWN")
        after = df_filled['code'].isnull().sum()
        print(
            f"   ✓ code: {before} NaN → {after} NaN "
            f"(filled with {city_name.upper()}_UNKNOWN)"
        )

    # id
    if 'id' in df_filled.columns:
        before = df_filled['id'].isnull().sum()
        df_filled['id'] = df_filled['id'].fillna(-1).astype(int)
        after = df_filled['id'].isnull().sum()
        print(f"   ✓ id: {before} NaN → {after} NaN (filled with -1)")

    # --- Drop baris yang seluruh kolomnya NaN ---
    before_rows = len(df_filled)
    df_filled = df_filled.dropna(how='all')
    after_rows = len(df_filled)
    if before_rows != after_rows:
        print(f"   ✓ Dropped {before_rows - after_rows} all-NaN rows")
    remaining_nan = df_filled.isnull().sum().sum()
    print(f"   📊 Total NaN tersisa: {remaining_nan}")
    return df_filled


data_ancona = handle_missing_values(data_ancona, 'Ancona')
data_athens = handle_missing_values(data_athens, 'Athens')
data_zaragoza = handle_missing_values(data_zaragoza, 'Zaragoza')

print("\n✅ Missing values handled")


🔧 Processing Ancona...
   ✓ temperature: 0 NaN → 0 NaN (ffill + interpolate)
   ✓ relative_humidity: 0 NaN → 0 NaN (ffill + interpolate)
   ✓ dewpoint_temp: 0 NaN → 0 NaN (ffill + interpolate)
   ✓ soil_temp: 0 NaN → 0 NaN (ffill + interpolate)
   ✓ wind_u: 0 NaN → 0 NaN (ffill + interpolate)
   ✓ wind_v: 0 NaN → 0 NaN (ffill + interpolate)
   ✓ pm25: 45118 NaN → 0 NaN (linear interpolate)
   ✓ pm10: 27859 NaN → 0 NaN (linear interpolate)
   ✓ no2: 7778 NaN → 0 NaN (linear interpolate)
   ✓ o3: 24803 NaN → 0 NaN (linear interpolate)
   ✓ precipitation: 0 NaN → 0 NaN (fill with 0)
   ✓ vegetation_high: 0 NaN → 0 NaN (filled with median = 3.0575)
   ✓ vegetation_low: 0 NaN → 0 NaN (filled with median = 2.7762)
   ✓ code: 0 NaN → 0 NaN (filled with ANCONA_UNKNOWN)
   ✓ id: 0 NaN → 0 NaN (filled with -1)
   📊 Total NaN tersisa: 0

🔧 Processing Athens...
   ✓ temperature: 7565 NaN → 0 NaN (ffill + interpolate)
   ✓ relative_humidity: 8573 NaN → 0 NaN (ffill + interpolate)
   ✓ dewpoint_tem

In [ ]:
print("\n=== Remaining Missing Values ===")
for city_name, df in [
    ("Ancona", data_ancona),
    ("Athens", data_athens),
    ("Zaragoza", data_zaragoza)
]:
    print(f"\n{city_name}")
    remaining = df.isnull().sum()
    remaining = remaining[remaining > 0]
    print(remaining if len(remaining) > 0 else "No missing values")


=== Remaining Missing Values ===

Ancona
No missing values

Athens
No missing values

Zaragoza
No missing values


In [ ]:
print(data_ancona.shape)
print(data_athens.shape)
print(data_zaragoza.shape)

(417626, 27)
(1726464, 27)
(238336, 26)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238336 entries, 0 to 238335
Data columns (total 26 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   date               238336 non-null  datetime64[ns]
 1   no2                238336 non-null  float64       
 2   o3                 238336 non-null  float64       
 3   pm10               238336 non-null  float64       
 4   latitude           238336 non-null  float64       
 5   longitude          238336 non-null  float64       
 6   station_name       238336 non-null  object        
 7   wind_u             238336 non-null  float64       
 8   wind_v             238336 non-null  float64       
 9   dewpoint_temp      238336 non-null  float64       
 10  temperature        238336 non-null  float64       
 11  vegetation_high    238336 non-null  float64       
 12  vegetation_low     238336 non-null  float64       
 13  soil_temp          238336 non-null  float64 

In [ ]:
# def handle_missing_values(df, city_name):
#     print(f"\n🔧 Processing {city_name}...")
#     df_filled = df.copy()

#     # --- Meteorologi: ffill max 24h + linear interpolation ---
#     meteo_cols = ['temperature', 'relative_humidity', 'dewpoint_temp', 'soil_temp']
#     for col in meteo_cols:
#         if col not in df_filled.columns:
#             continue
#         before = df_filled[col].isnull().sum()
#         # FIX: ffill() bukan fillna(method='ffill') — sudah deprecated
#         df_filled[col] = df_filled[col].ffill(limit=24)
#         df_filled[col] = df_filled[col].interpolate(method='linear', limit_direction='both')
#         after = df_filled[col].isnull().sum()
#         print(f"   ✓ {col}: {before} NaN → {after} NaN (ffill + interpolate)")

#     # --- Wind: ffill + interpolation ---
#     for col in ['wind_u', 'wind_v']:
#         if col not in df_filled.columns:
#             continue
#         before = df_filled[col].isnull().sum()
#         df_filled[col] = df_filled[col].ffill(limit=24)
#         df_filled[col] = df_filled[col].interpolate(method='linear', limit_direction='both')
#         after = df_filled[col].isnull().sum()
#         print(f"   ✓ {col}: {before} NaN → {after} NaN (ffill + interpolate)")

#     # --- Polutan: linear interpolation ---
#     # FIX: Skip PM2.5 Zaragoza — kalau semua NaN berarti data memang tidak tersedia
#     pollutant_cols = ['pm25', 'pm10', 'no2', 'o3']
#     for col in pollutant_cols:
#         if col not in df_filled.columns:
#             continue

#         before = df_filled[col].isnull().sum()

#         # FIX: Kalau semua NaN → data tidak tersedia → SKIP
#         if df_filled[col].isnull().all():
#             print(f"   ⚠  {col}: semua NaN (data tidak tersedia di {city_name}) → SKIP")
#             continue

#         df_filled[col] = df_filled[col].interpolate(method='linear', limit_direction='both')
#         after = df_filled[col].isnull().sum()
#         print(f"   ✓ {col}: {before} NaN → {after} NaN (linear interpolate)")

#     # --- Precipitation: fill 0 ---
#     if 'precipitation' in df_filled.columns:
#         before = df_filled['precipitation'].isnull().sum()
#         df_filled['precipitation'] = df_filled['precipitation'].fillna(0)
#         print(f"   ✓ precipitation: {before} NaN → 0 (fill with 0)")

#     # --- Vegetation: fill median jika < 50% missing ---
#     for col in ['vegetation_high', 'vegetation_low']:
#         if col not in df_filled.columns:
#             continue
#         missing_pct = (df_filled[col].isnull().sum() / len(df_filled)) * 100
#         if missing_pct < 50:
#             median_val = df_filled[col].median()
#             before = df_filled[col].isnull().sum()
#             df_filled[col] = df_filled[col].fillna(median_val)
#             print(f"   ✓ {col}: {before} NaN → filled with median ({median_val:.4f})")
#         else:
#             print(f"   ⚠  {col}: {missing_pct:.1f}% missing → terlalu banyak, skip")

#     # Drop baris yang semua nilainya NaN
#     before_rows = len(df_filled)
#     df_filled = df_filled.dropna(how='all')
#     after_rows = len(df_filled)
#     if before_rows != after_rows:
#         print(f"   ✓ Dropped {before_rows - after_rows} all-NaN rows")

#     remaining_nan = df_filled.isnull().sum().sum()
#     print(f"   📊 Total NaN tersisa: {remaining_nan}")

#     return df_filled

# data_ancona   = handle_missing_values(data_ancona,   'Ancona')
# data_athens   = handle_missing_values(data_athens,   'Athens')
# data_zaragoza = handle_missing_values(data_zaragoza, 'Zaragoza')

# print("\n✅ Missing values handled")

In [ ]:
# data_athens['code'] = data_athens['code'].fillna('ATH_UNKNOWN')

# CELL 11: REMOVE DUPLICATES

In [ ]:
def remove_duplicates(df, city_name):
    print(f"\n🔍 Checking duplicates in {city_name}...")
    initial_rows = len(df)

    exact_dups = df.duplicated().sum()
    if exact_dups > 0:
        df = df.drop_duplicates()
        print(f"   ✓ Removed {exact_dups} exact duplicates")
    else:
        print(f"   ✓ No exact duplicates found")

    temporal_dups = df.duplicated(subset=['date', 'latitude', 'longitude'], keep='first').sum()
    if temporal_dups > 0:
        df = df.drop_duplicates(subset=['date', 'latitude', 'longitude'], keep='first')
        print(f"   ✓ Removed {temporal_dups} temporal duplicates")
    else:
        print(f"   ✓ No temporal duplicates found")

    removed = initial_rows - len(df)
    print(f"   📊 Total removed: {removed} | Remaining: {len(df):,}")
    return df

data_ancona   = remove_duplicates(data_ancona,   'Ancona')
data_athens   = remove_duplicates(data_athens,   'Athens')
data_zaragoza = remove_duplicates(data_zaragoza, 'Zaragoza')

print("\n✅ Duplicates removed")


🔍 Checking duplicates in Ancona...
   ✓ No exact duplicates found
   ✓ No temporal duplicates found
   📊 Total removed: 0 | Remaining: 417,626

🔍 Checking duplicates in Athens...
   ✓ No exact duplicates found
   ✓ No temporal duplicates found
   📊 Total removed: 0 | Remaining: 1,726,464

🔍 Checking duplicates in Zaragoza...
   ✓ No exact duplicates found
   ✓ No temporal duplicates found
   📊 Total removed: 0 | Remaining: 238,336

✅ Duplicates removed


# CELL 12: OUTLIER DETECTION & FLAGGING

FIX: Guard all-NaN kolom (Zaragoza PM2.5)

In [ ]:
def detect_outliers(df, city_name):
    print(f"\n🔍 Detecting outliers in {city_name}...")
    df = df.copy()
    df['outlier_flag']   = 0
    df['outlier_method'] = ''

    numeric_cols = ['pm25', 'pm10', 'no2', 'o3', 'temperature', 'relative_humidity']

    for col in numeric_cols:
        if col not in df.columns:
            continue

        # FIX: Skip kalau semua NaN (Zaragoza PM2.5)
        if df[col].isnull().all():
            print(f"   ⚠  {col}: semua NaN → skip outlier detection")
            continue

        valid_count = df[col].dropna().shape[0]
        if valid_count < 10:
            print(f"   ⚠  {col}: data valid terlalu sedikit ({valid_count}) → skip")
            continue

        # Z-score method (|z| > 3)
        mean_val = df[col].mean()
        std_val  = df[col].std()
        if std_val == 0:
            continue
        z_scores  = np.abs((df[col] - mean_val) / std_val)
        z_outliers = z_scores > 3
        df.loc[z_outliers, 'outlier_flag']   = 1
        df.loc[z_outliers, 'outlier_method'] = (
            df.loc[z_outliers, 'outlier_method'] + f'zscore_{col}|'
        )
        print(f"   ✓ {col}: {z_outliers.sum()} Z-score outliers (|z|>3)")

        # IQR method
        Q1  = df[col].quantile(0.25)
        Q3  = df[col].quantile(0.75)
        IQR = Q3 - Q1
        iqr_outliers = (df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)
        df.loc[iqr_outliers, 'outlier_flag']   = 1
        df.loc[iqr_outliers, 'outlier_method'] = (
            df.loc[iqr_outliers, 'outlier_method'] + f'iqr_{col}|'
        )
        print(f"   ✓ {col}: {iqr_outliers.sum()} IQR outliers")

    # Domain rules
    if 'pm25' in df.columns and not df['pm25'].isnull().all():
        neg_pm25 = df['pm25'] < 0
        df.loc[neg_pm25, 'outlier_flag'] = 1
        print(f"   ✓ Negative PM2.5: {neg_pm25.sum()} flagged")

    if 'pm10' in df.columns and not df['pm10'].isnull().all():
        neg_pm10 = df['pm10'] < 0
        df.loc[neg_pm10, 'outlier_flag'] = 1
        print(f"   ✓ Negative PM10: {neg_pm10.sum()} flagged")

    if 'temperature' in df.columns:
        extreme_temp = df['temperature'] > 60
        df.loc[extreme_temp, 'outlier_flag'] = 1
        print(f"   ✓ Extreme temp (>60°C): {extreme_temp.sum()} flagged")

    total_out = (df['outlier_flag'] == 1).sum()
    print(f"   📊 Total outliers flagged: {total_out} ({total_out/len(df)*100:.2f}%)")
    return df

data_ancona   = detect_outliers(data_ancona,   'Ancona')
data_athens   = detect_outliers(data_athens,   'Athens')
data_zaragoza = detect_outliers(data_zaragoza, 'Zaragoza')

print("\n✅ Outliers detected and flagged (NOT deleted)")


🔍 Detecting outliers in Ancona...
   ✓ pm25: 6473 Z-score outliers (|z|>3)
   ✓ pm25: 17742 IQR outliers
   ✓ pm10: 5679 Z-score outliers (|z|>3)
   ✓ pm10: 15301 IQR outliers
   ✓ no2: 7575 Z-score outliers (|z|>3)
   ✓ no2: 22723 IQR outliers
   ✓ o3: 203 Z-score outliers (|z|>3)
   ✓ o3: 447 IQR outliers
   ✓ temperature: 0 Z-score outliers (|z|>3)
   ✓ temperature: 75617 IQR outliers
   ✓ relative_humidity: 242 Z-score outliers (|z|>3)
   ✓ relative_humidity: 220 IQR outliers
   ✓ Negative PM2.5: 0 flagged
   ✓ Negative PM10: 0 flagged
   ✓ Extreme temp (>60°C): 211160 flagged
   📊 Total outliers flagged: 306703 (73.44%)

🔍 Detecting outliers in Athens...
   ✓ pm25: 19420 Z-score outliers (|z|>3)
   ✓ pm25: 111327 IQR outliers
   ✓ pm10: 29420 Z-score outliers (|z|>3)
   ✓ pm10: 110560 IQR outliers
   ✓ no2: 24655 Z-score outliers (|z|>3)
   ✓ no2: 60140 IQR outliers
   ✓ o3: 3166 Z-score outliers (|z|>3)
   ✓ o3: 5523 IQR outliers
   ✓ temperature: 256 Z-score outliers (|z|>3)
  

# CELL 13: VISUALISASI OUTLIERS (TAMPIL DI COLAB + SIMPAN KE DRIVE)

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'PM Distribution (PM2.5 atau PM10)',
        'NO₂ Distribution',
        'O₃ Distribution',
        'Temperature Distribution'
    ),
    specs=[[{'type': 'box'}, {'type': 'box'}],
           [{'type': 'box'}, {'type': 'box'}]]
)

cities_data  = [data_ancona, data_athens, data_zaragoza]
cities_names = ['Ancona', 'Athens', 'Zaragoza']
city_colors  = {'Ancona': '#3498DB', 'Athens': '#E74C3C', 'Zaragoza': '#2ECC71'}

# Row 1 Col 1: PM (PM2.5 jika tersedia, fallback ke PM10)
for df, city in zip(cities_data, cities_names):
    if 'pm25' in df.columns and not df['pm25'].isnull().all():
        col_use  = 'pm25'
        label    = 'PM2.5'
    elif 'pm10' in df.columns:
        col_use  = 'pm10'
        label    = 'PM10'
    else:
        continue

    fig.add_trace(
        go.Box(
            y=df[col_use],
            name=f'{city} ({label})',
            marker=dict(color=city_colors[city]),
            boxmean=True,
            hovertemplate=f'<b>{city} {label}</b><br>Value: %{{y:.1f}}<extra></extra>'
        ),
        row=1, col=1
    )

# Row 1 Col 2: NO2
for df, city in zip(cities_data, cities_names):
    if 'no2' not in df.columns:
        continue
    fig.add_trace(
        go.Box(
            y=df['no2'], name=city,
            marker=dict(color=city_colors[city]),
            boxmean=True, showlegend=False,
            hovertemplate=f'<b>{city} NO₂</b><br>Value: %{{y:.1f}}<extra></extra>'
        ),
        row=1, col=2
    )

# Row 2 Col 1: O3
for df, city in zip(cities_data, cities_names):
    if 'o3' not in df.columns:
        continue
    fig.add_trace(
        go.Box(
            y=df['o3'], name=city,
            marker=dict(color=city_colors[city]),
            boxmean=True, showlegend=False,
            hovertemplate=f'<b>{city} O₃</b><br>Value: %{{y:.1f}}<extra></extra>'
        ),
        row=2, col=1
    )

# Row 2 Col 2: Temperature
for df, city in zip(cities_data, cities_names):
    if 'temperature' not in df.columns:
        continue
    fig.add_trace(
        go.Box(
            y=df['temperature'], name=city,
            marker=dict(color=city_colors[city]),
            boxmean=True, showlegend=False,
            hovertemplate=f'<b>{city} Temp</b><br>Value: %{{y:.1f}}°C<extra></extra>'
        ),
        row=2, col=2
    )

fig.update_yaxes(title_text='PM Concentration (µg/m³)', row=1, col=1)
fig.update_yaxes(title_text='NO₂ (µg/m³)',              row=1, col=2)
fig.update_yaxes(title_text='O₃ (µg/m³)',               row=2, col=1)
fig.update_yaxes(title_text='Temperature (°C)',          row=2, col=2)

fig.update_layout(
    title_text='Outlier Detection via Box Plot — All Cities',
    title_font_size=20,
    title_font_color='#1f77b4',
    height=700,
    plot_bgcolor='rgba(240,240,240,0.5)',
    paper_bgcolor='white',
    hovermode='closest',
    font=dict(size=11),
    legend=dict(orientation='h', y=-0.08, x=0.5, xanchor='center')
)

fig.write_html(f'{RESULTS_PREPROCESSING}/02_outliers_boxplot.html')
print("✅ Saved: 02_outliers_boxplot.html")
fig.show()  # Tampil di Colab output

✅ Saved: 02_outliers_boxplot.html
Buffered data was truncated after reaching the output size limit.

# CELL 14: MERGE DATASETS

FIX: Handle NaN di code & id setelah concat

In [ ]:
all_cities = pd.concat(
    [data_ancona, data_athens, data_zaragoza],
    ignore_index=True
)
all_cities = all_cities.sort_values('date').reset_index(drop=True)

In [ ]:
# FIX: Handle NaN di kolom identifier setelah merge
if 'code' in all_cities.columns:
    all_cities['code'] = all_cities['code'].fillna('UNKNOWN')

if 'id' in all_cities.columns:
    all_cities['id'] = all_cities['id'].fillna(-1).astype(int)

print(f"✅ Datasets merged")
print(f"   Total rows   : {len(all_cities):,}")
print(f"   Date range   : {all_cities['date'].min()} → {all_cities['date'].max()}")
print(f"   Cities       : {all_cities['city'].unique().tolist()}")
print(f"   Total columns: {len(all_cities.columns)}")
print(f"\n   NaN tersisa di 'code': {all_cities['code'].isnull().sum()}")
print(f"   NaN tersisa di 'id'  : {all_cities['id'].isnull().sum()}")

# Verifikasi PM2.5 per kota setelah merge
print(f"\n📋 PM2.5 STATUS SETELAH MERGE:")
for city in ['Ancona', 'Athens', 'Zaragoza']:
    df_c   = all_cities[all_cities['city'] == city]
    n_nan  = df_c['pm25'].isnull().sum() if 'pm25' in all_cities.columns else 'N/A'
    n_tot  = len(df_c)
    avail  = df_c['pm25_available'].iloc[0]
    print(f"   {city:<10}: pm25_available={avail}, NaN={n_nan}/{n_tot}")

✅ Datasets merged
   Total rows   : 2,382,426
   Date range   : 2020-05-01 00:00:00 → 2023-10-31 23:00:00
   Cities       : ['Athens', 'Zaragoza', 'Ancona']
   Total columns: 29

   NaN tersisa di 'code': 0
   NaN tersisa di 'id'  : 0

📋 PM2.5 STATUS SETELAH MERGE:
   Ancona    : pm25_available=True, NaN=0/417626
   Athens    : pm25_available=True, NaN=0/1726464
   Zaragoza  : pm25_available=False, NaN=238336/238336


In [ ]:
print(all_cities.shape)

(2382426, 29)


# CELL 15: SAVE PROCESSED & CLEANED DATASETS

In [ ]:
# Individual city files
data_ancona.to_csv(f'{PROCESSED_DATA_PATH}/ancona_processed.csv',   index=False)
data_athens.to_csv(f'{PROCESSED_DATA_PATH}/athens_processed.csv',   index=False)
data_zaragoza.to_csv(f'{PROCESSED_DATA_PATH}/zaragoza_processed.csv', index=False)
all_cities.to_csv(f'{PROCESSED_DATA_PATH}/all_cities_merged.csv',   index=False)

# Cleaned dataset dengan anomaly placeholder
all_cities['anomaly_flag']   = 0
all_cities['anomaly_score']  = 0.0
all_cities['anomaly_method'] = ''

# Tandai outliers sebagai potential anomalies
all_cities.loc[all_cities['outlier_flag'] == 1, 'anomaly_flag'] = 1

all_cities.to_csv(
    f'{CLEANED_DATA_PATH}/all_cities_cleaned_with_anomalies.csv',
    index=False
)

print("✅ Files saved:")
print(f"   • {PROCESSED_DATA_PATH}/ancona_processed.csv")
print(f"   • {PROCESSED_DATA_PATH}/athens_processed.csv")
print(f"   • {PROCESSED_DATA_PATH}/zaragoza_processed.csv")
print(f"   • {PROCESSED_DATA_PATH}/all_cities_merged.csv")
print(f"   • {CLEANED_DATA_PATH}/all_cities_cleaned_with_anomalies.csv")

✅ Files saved:
   • /content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities/data/processed/ancona_processed.csv
   • /content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities/data/processed/athens_processed.csv
   • /content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities/data/processed/zaragoza_processed.csv
   • /content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities/data/processed/all_cities_merged.csv
   • /content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities/data/cleaned/all_cities_cleaned_with_anomalies.csv


# CELL 16: DATA QUALITY SUMMARY REPORT

In [ ]:
quality_data = {
    'Metric'    : [],
    'Ancona'    : [],
    'Athens'    : [],
    'Zaragoza'  : [],
    'All Cities': []
}

datasets      = [data_ancona, data_athens, data_zaragoza, all_cities]
dataset_names = ['Ancona', 'Athens', 'Zaragoza', 'All Cities']

# Total rows
quality_data['Metric'].append('Total Rows')
for df, name in zip(datasets, dataset_names):
    quality_data[name].append(f"{len(df):,}")

# Remaining NaN
quality_data['Metric'].append('Remaining NaN (total)')
for df, name in zip(datasets, dataset_names):
    quality_data[name].append(f"{df.isnull().sum().sum():,}")

# PM2.5 NaN
quality_data['Metric'].append('PM2.5 NaN')
for df, name in zip(datasets, dataset_names):
    val = df['pm25'].isnull().sum() if 'pm25' in df.columns else 'N/A'
    quality_data[name].append(str(val))

# Outliers flagged
quality_data['Metric'].append('Outliers Flagged')
for df, name in zip(datasets, dataset_names):
    val = (df['outlier_flag'] == 1).sum() if 'outlier_flag' in df.columns else 0
    quality_data[name].append(f"{val:,}")

# Date range
quality_data['Metric'].append('Date Range')
for df, name in zip(datasets, dataset_names):
    quality_data[name].append(
        f"{df['date'].min().date()} → {df['date'].max().date()}"
    )

# PM2.5 available
quality_data['Metric'].append('PM2.5 Available')
for df, name in zip(datasets, dataset_names):
    if 'pm25_available' in df.columns:
        val = str(df['pm25_available'].iloc[0])
    else:
        val = 'N/A'
    quality_data[name].append(val)

quality_df = pd.DataFrame(quality_data)
quality_df.to_csv(f'{RESULTS_PREPROCESSING}/data_quality_report.csv', index=False)

print("📊 DATA QUALITY SUMMARY:")
print(quality_df.to_string(index=False))
print(f"\n✅ Saved: data_quality_report.csv")

📊 DATA QUALITY SUMMARY:
               Metric                  Ancona                  Athens                Zaragoza              All Cities
           Total Rows                 417,626               1,726,464                 238,336               2,382,426
Remaining NaN (total)                       0                       0                       0                 238,336
            PM2.5 NaN                       0                       0                     N/A                  238336
     Outliers Flagged                 306,703                 204,680                  19,785                 531,168
           Date Range 2021-09-01 → 2023-10-31 2020-05-01 → 2023-05-29 2020-08-01 → 2023-05-29 2020-05-01 → 2023-10-31
      PM2.5 Available                    True                    True                   False                    True

✅ Saved: data_quality_report.csv


# CELL 17: VISUALISASI DATA OVERVIEW (TAMPIL DI COLAB + SIMPAN KE DRIVE)

FIX: Missing values dihitung dari data asli (bukan hardcode)

In [ ]:
# Hitung missing values dari raw data
raw_ancona   = pd.read_csv(f'{RAW_DATA_PATH}/ancona_data.csv')
raw_athens   = pd.read_csv(f'{RAW_DATA_PATH}/athens_data.csv')
raw_zaragoza = pd.read_csv(f'{RAW_DATA_PATH}/zaragoza_data.csv')

missing_pct_before = [
    (raw_ancona.isnull().sum().sum()   / raw_ancona.size)   * 100,
    (raw_athens.isnull().sum().sum()   / raw_athens.size)   * 100,
    (raw_zaragoza.isnull().sum().sum() / raw_zaragoza.size) * 100,
]

cities      = ['Ancona', 'Athens', 'Zaragoza']
city_colors_list = ['#3498DB', '#E74C3C', '#2ECC71']

In [ ]:
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Row Count per City',
        'Outliers % per City',
        'Missing Values % (Before Handling)',
        'Data Distribution per Month',
        'PM Distribution per City',
        'Temperature Distribution per City'
    ),
    specs=[
        [{'type': 'bar'},       {'type': 'bar'}],
        [{'type': 'bar'},       {'type': 'histogram'}],
        [{'type': 'box'},       {'type': 'box'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.12
)

# 1. Row count
row_counts = [len(data_ancona), len(data_athens), len(data_zaragoza)]
fig.add_trace(
    go.Bar(
        x=cities, y=row_counts,
        marker=dict(color=city_colors_list),
        text=[f'{v:,}' for v in row_counts],
        textposition='outside',
        showlegend=False,
        hovertemplate='<b>%{x}</b><br>Rows: %{y:,}<extra></extra>'
    ),
    row=1, col=1
)
fig.update_yaxes(title_text='Row Count', row=1, col=1)

# 2. Outliers %
outliers_pct = [
    (data_ancona['outlier_flag'].sum()   / len(data_ancona))   * 100,
    (data_athens['outlier_flag'].sum()   / len(data_athens))   * 100,
    (data_zaragoza['outlier_flag'].sum() / len(data_zaragoza)) * 100,
]
fig.add_trace(
    go.Bar(
        x=cities, y=outliers_pct,
        marker=dict(color=['#E67E22', '#9B59B6', '#1ABC9C']),
        text=[f'{x:.2f}%' for x in outliers_pct],
        textposition='outside',
        showlegend=False,
        hovertemplate='<b>%{x}</b><br>Outliers: %{y:.2f}%<extra></extra>'
    ),
    row=1, col=2
)
fig.update_yaxes(title_text='Outliers %', row=1, col=2)

# 3. Missing values % BEFORE (FIX: dari data asli, bukan hardcode)
fig.add_trace(
    go.Bar(
        x=cities, y=missing_pct_before,
        marker=dict(color=['#FF6B6B', '#FFA07A', '#FFD700']),
        text=[f'{x:.1f}%' for x in missing_pct_before],
        textposition='outside',
        showlegend=False,
        hovertemplate='<b>%{x}</b><br>Missing: %{y:.1f}%<extra></extra>'
    ),
    row=2, col=1
)
fig.update_yaxes(title_text='Missing %', row=2, col=1)

# 4. Date distribution
for df, city, color in zip(
    [data_ancona, data_athens, data_zaragoza], cities, city_colors_list
):
    fig.add_trace(
        go.Histogram(
            x=df['date'], name=city,
            marker=dict(color=color),
            nbinsx=50, opacity=0.7,
            hovertemplate=f'<b>{city}</b><br>Date: %{{x}}<br>Count: %{{y}}<extra></extra>'
        ),
        row=2, col=2
    )
fig.update_xaxes(title_text='Date',  row=2, col=2)
fig.update_yaxes(title_text='Count', row=2, col=2)

# 5. PM distribution
for df, city, color in zip(
    [data_ancona, data_athens, data_zaragoza], cities, city_colors_list
):
    # Pakai PM2.5 kalau tersedia, fallback ke PM10
    if 'pm25' in df.columns and not df['pm25'].isnull().all():
        col_use = 'pm25'
        label   = 'PM2.5'
    else:
        col_use = 'pm10'
        label   = 'PM10'

    fig.add_trace(
        go.Box(
            y=df[col_use], name=f'{city} ({label})',
            marker=dict(color=color),
            boxmean=True, showlegend=False,
            hovertemplate=f'<b>{city} {label}</b><br>Value: %{{y:.1f}}<extra></extra>'
        ),
        row=3, col=1
    )
fig.update_yaxes(title_text='PM Concentration (µg/m³)', row=3, col=1)

# 6. Temperature distribution
for df, city, color in zip(
    [data_ancona, data_athens, data_zaragoza], cities, city_colors_list
):
    fig.add_trace(
        go.Box(
            y=df['temperature'], name=city,
            marker=dict(color=color),
            boxmean=True, showlegend=False,
            hovertemplate=f'<b>{city} Temp</b><br>Value: %{{y:.1f}}°C<extra></extra>'
        ),
        row=3, col=2
    )
fig.update_yaxes(title_text='Temperature (°C)', row=3, col=2)

fig.update_layout(
    title_text='Data Preprocessing & Quality Overview — All Cities',
    title_font_size=22,
    title_font_color='#2C3E50',
    height=1100,
    showlegend=True,
    plot_bgcolor='rgba(245,245,245,0.7)',
    paper_bgcolor='white',
    hovermode='closest',
    font=dict(size=11),
    legend=dict(orientation='h', y=-0.05, x=0.5, xanchor='center'),
    margin=dict(l=80, r=60, t=100, b=80)
)

fig.write_html(f'{RESULTS_PREPROCESSING}/03_data_quality_overview.html')
print("✅ Saved: 03_data_quality_overview.html")
fig.show()  # Tampil di Colab output

✅ Saved: 03_data_quality_overview.html


# CELL 18: DESCRIPTIVE STATISTICS

In [ ]:
summary_cols = ['pm25', 'pm10', 'no2', 'o3', 'temperature', 'relative_humidity']

for city_name, df in [
    ('ANCONA',   data_ancona),
    ('ATHENS',   data_athens),
    ('ZARAGOZA', data_zaragoza)
]:
    available_cols = [c for c in summary_cols if c in df.columns and not df[c].isnull().all()]
    print(f"\n📊 {city_name} - DESCRIPTIVE STATISTICS:")
    print(df[available_cols].describe().round(2).to_string())


📊 ANCONA - DESCRIPTIVE STATISTICS:
            pm25       pm10        no2         o3  temperature  relative_humidity
count  417626.00  417626.00  417626.00  417626.00    417626.00          417626.00
mean       13.01      18.86      12.80      56.98        55.23              75.30
std         7.79      10.39       9.43      25.82        18.21              18.55
min         0.00       0.00       0.00       0.00        10.11              12.94
25%         7.87      11.94       6.42      38.15        52.02              62.29
50%        11.19      16.53      10.31      56.51        60.13              79.91
75%        16.15      23.46      16.14      74.79        66.98              90.89
max       200.00     249.00     285.00     162.00        88.42             100.00

📊 ATHENS - DESCRIPTIVE STATISTICS:
             pm25        pm10         no2          o3  temperature  relative_humidity
count  1726464.00  1726464.00  1726464.00  1726464.00   1726464.00         1726464.00
mean        15.07 

# CELL 19: FINAL SUMMARY

In [ ]:
print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║        NOTEBOOK 01 — PREPROCESSING & CLEANING COMPLETE ✅           ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  BUG FIXES:                                                          ║
║  ✓ pio.renderers → 'colab' (chart tampil di Colab output)          ║
║  ✓ PM2.5 Zaragoza → TIDAK diinterpolasi (data memang tidak ada)    ║
║  ✓ code/id Zaragoza → diisi placeholder ('ZGZ_UNKNOWN' / -1)       ║
║  ✓ pm25_available flag → True/False per kota                       ║
║  ✓ ffill()/bfill() → tidak pakai method= yang deprecated           ║
║  ✓ detect_outliers → guard all-NaN kolom                           ║
║  ✓ Missing values viz → dihitung dari raw data (bukan hardcode)    ║
║                                                                      ║
║  PENTING UNTUK NOTEBOOK LAIN (02-09):                               ║
║  ✓ PATH harus: Project Mandiri/Air Quality Monitoring in...        ║
║  ✓ Gunakan pm25_available flag untuk skip Zaragoza di PM2.5        ║
║  ✓ Pattern: if city == 'Zaragoza' and pollutant == 'pm25': skip   ║
║                                                                      ║
║  FILES SAVED:                                                        ║
║  ✓ processed/ancona_processed.csv                                  ║
║  ✓ processed/athens_processed.csv                                  ║
║  ✓ processed/zaragoza_processed.csv                                ║
║  ✓ processed/all_cities_merged.csv                                 ║
║  ✓ cleaned/all_cities_cleaned_with_anomalies.csv                   ║
║  ✓ preprocessing/01_missing_values_analysis.html                   ║
║  ✓ preprocessing/02_outliers_boxplot.html                          ║
║  ✓ preprocessing/03_data_quality_overview.html                     ║
║  ✓ preprocessing/data_quality_report.csv                           ║
║                                                                      ║
║  NEXT → Notebook 02: EDA Analysis                                   ║
╚══════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════╗
║        NOTEBOOK 01 — PREPROCESSING & CLEANING COMPLETE ✅           ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  BUG FIXES:                                                          ║
║  ✓ pio.renderers → 'colab' (chart tampil di Colab output)          ║
║  ✓ PM2.5 Zaragoza → TIDAK diinterpolasi (data memang tidak ada)    ║
║  ✓ code/id Zaragoza → diisi placeholder ('ZGZ_UNKNOWN' / -1)       ║
║  ✓ pm25_available flag → True/False per kota                       ║
║  ✓ ffill()/bfill() → tidak pakai method= yang deprecated           ║
║  ✓ detect_outliers → guard all-NaN kolom                           ║
║  ✓ Missing values viz → dihitung dari raw data (bukan hardcode)    ║
║                                                                      ║
║  PENTING UNTUK NOTEBOOK LAIN (02-09):                           

# VISUALISASI

In [ ]:
import os

files = [
    "01_missing_values_analysis.html",
    "02_outliers_boxplot.html",
    "03_data_quality_overview.html"
]

base = "/content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities/results/preprocessing"

for f in files:
    size = os.path.getsize(f"{base}/{f}") / (1024 * 1024)
    print(f"{f} : {size:.2f} MB")

In [ ]:
from IPython.display import HTML

HTML(filename="/content/drive/MyDrive/Project Mandiri/Air Quality Monitoring in European Cities/results/preprocessing/01_missing_values_analysis.html")